# DOTM — извлечение загрузочных ресурсов игры
Этот ноутбук использует **твой собственный ISO на Google Drive**, ничего не скачивает из пиратских источников и не публикует игровые файлы на GitHub.
1. Нажми **Среда выполнения → Выполнить все**.
2. Разреши доступ к Google Drive.
3. Дождись сообщения `SUCCESS` и напиши в ChatGPT **ГОТОВО**.

Извлекает ISO только во временное хранилище Colab. Загружает на Drive один небольшой ZIP в `MyDrive/DOTM_PORT_RESEARCH/DOTM_BOOT_RESOURCES.zip` (исходный ISO не изменяется).


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, shutil, hashlib, subprocess, zipfile, time

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive")
iso = root / "Transformers - Dark of the Moon (USA) (En,Fr,De,Es,It).iso"
out = root / "DOTM_PORT_RESEARCH"
out.mkdir(exist_ok=True)
if not iso.is_file() or iso.stat().st_size != 7838695424:
    raise RuntimeError("Original DOTM ISO missing or unexpected size. No changes were made.")

# Original assets only. Do not upload these to the public GitHub repository.
files = [
    "TransGame/Xbox360TOC.txt",
    "TransGame/Config/Xenon/Cooked/Coalesced.ini",
    "TransGame/Localization/Cooked/Xenon/Coalesced.int",
    "TransGame/CookedXenon/Core.xxx",
    "TransGame/CookedXenon/Engine.xxx",
    "TransGame/CookedXenon/HM_Engine.xxx",
    "TransGame/CookedXenon/EngineFonts.xxx",
    "TransGame/CookedXenon/EngineMaterials.xxx",
    "TransGame/CookedXenon/EngineResources.xxx",
    "TransGame/CookedXenon/Engine_MI_Shaders.xxx",
    "TransGame/CookedXenon/GFxUI.xxx",
    "TransGame/CookedXenon/IpDrv.xxx",
    "TransGame/CookedXenon/TransGame.xxx",
    "TransGame/CookedXenon/Startup_int.xxx",
    "TransGame/CookedXenon/UI_FrontEnd_m.xxx",
    "TransGame/CookedXenon/UI_FrontEnd_m_LOC_int.xxx",
    "TransGame/CookedXenon/UI_GFxFrontEnd_p.xxx",
    "TransGame/CookedXenon/UI_GFxShared_p.xxx",
    "TransGame/CookedXenon/UI_GFxLoading_p.xxx",
    "TransGame/CookedXenon/UI_GFxFonts_p.xxx",
]
zip_path = out / "DOTM_BOOT_RESOURCES.zip"
if zip_path.is_file() and zip_path.stat().st_size > 1000000:
    with zipfile.ZipFile(zip_path) as z:
        if all(name in z.namelist() for name in files):
            print("Already extracted:", zip_path, zip_path.stat().st_size, "bytes")
            print("You can now send this chat the Drive file link or simply say: ГОТОВО")
            sys.exit(0)

source = Path("/content/extract-xiso")
if not (source / "CMakeLists.txt").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/XboxDev/extract-xiso.git", str(source)], check=True)
subprocess.run(["cmake", "-S", str(source), "-B", str(source / "build"), "-DCMAKE_BUILD_TYPE=Release"], check=True)
subprocess.run(["cmake", "--build", str(source / "build"), "-j", "2"], check=True)
exe = next((p for p in (source/"build").rglob("extract-xiso") if p.is_file() and os.access(p,os.X_OK)),None)
if exe is None:
    raise RuntimeError("extract-xiso was not built")

target = Path("/content/dotm_extracted")
if shutil.disk_usage("/content").free < 7*1024**3:
    raise RuntimeError("Colab needs at least 7 GiB free temporary space for extraction. Do not delete your ISO.")
target.mkdir(parents=True, exist_ok=True)

start = time.time()
print("Extracting original Xbox 360 game contents to temporary Colab storage...")
subprocess.run([str(exe), "-x", str(iso), "-d", str(target)], check=True)
print("Extraction finished in", round(time.time()-start,1),"seconds")

# Extract-xiso may put files in either the supplied directory or one child.
def locate(rel):
    direct = target / rel
    if direct.is_file():
        return direct
    for d in target.iterdir():
        p=d/rel
        if p.is_file():
            return p
    raise FileNotFoundError("Extracted ISO does not contain: "+rel)

details=[]
for rel in files:
    p=locate(rel)
    size=p.stat().st_size
    h=hashlib.sha256()
    with p.open("rb") as f:
        for buf in iter(lambda:f.read(4*1024*1024),b""):
            h.update(buf)
    details.append({"path":rel,"bytes":size,"sha256":h.hexdigest()})
    print(rel, size,"bytes")

temp_zip=out/"DOTM_BOOT_RESOURCES.uploading.zip"
with zipfile.ZipFile(temp_zip,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
    for rel in files:
        z.write(locate(rel),arcname=rel)
    z.writestr("DOTM_BOOT_RESOURCES_HASHES.json",json.dumps({"iso":iso.name,"files":details},indent=2))
with zipfile.ZipFile(temp_zip) as z:
    corrupt=z.testzip()
    if corrupt is not None:
        raise RuntimeError("ZIP verification failed: "+corrupt)
if temp_zip.stat().st_size > 240*1024**2:
    raise RuntimeError("Staging ZIP exceeds safe download size: "+str(temp_zip.stat().st_size))
temp_zip.replace(zip_path)
print("\nSUCCESS: boot resources saved to",zip_path)
print("ZIP size:",zip_path.stat().st_size,"bytes")
print("Files:",len(files))
print("Now return to ChatGPT and send: ГОТОВО")
